# Training a causal language model from scratch

- Up until now, we’ve mostly been using pretrained models and fine-tuning them for new use cases by reusing the weights from pretraining.
- This is commonly referred to as _transfer learning_, and it’s a very successful strategy for applying Transformer models to most real-world use cases where labeled data is sparse.
- In this notebook, we’ll take a different approach and train a completely new model from scratch.
    - This is a good approach to take if you have a lot of data and it is very different from the pretraining data used for the available models.
    - However, it also requires considerably more compute resources to pretrain a language model than just to fine-tune an existing one.
- Examples where it can make sense to train a new model include for datasets consisting of
    - musical notes,
    - molecular sequences such as DNA,
    - or programming languages
        - which have recently gained traction thanks to tools such as TabNine and GitHub’s Copilot, powered by OpenAI’s Codex model, that can generate long sequences of code.
        - This task of text generation is best addressed with auto-regressive or causal language models such as GPT-2.
- In this notebook we will build a scaled-down version of a code generation model:
    - we’ll focus on one-line completions instead of full functions or classes, using a subset of Python code.
    - When working with data in Python you are in frequent contact with the Python data science stack, consisting of the `matplotlib`, `seaborn`, `pandas`, and `scikit-learn` libraries.
    - When using those frameworks it’s common to need to look up specific commands, so it would be nice if we could use a model to complete these calls for us.

# Load deps

In [ ]:
# !pip install -q evaluate

In [ ]:
import os, torch, evaluate
from datasets import load_dataset, Dataset, DatasetDict
from kaggle_secrets import UserSecretsClient
from transformers import AutoTokenizer
from collections import defaultdict
from tqdm import tqdm
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForLanguageModeling
from transformers import AutoTokenizer, GPT2LMHeadModel, AutoConfig

# HF Login

In [ ]:
secret_label = "HF_TOKEN"
os.environ[secret_label] = UserSecretsClient().get_secret(secret_label)
!hf auth whoami

# Config

In [ ]:
hf_user_name = "amin-oj"
dataset_dict = dict(path = "huggingface-course/codeparrot-ds")
train_ds_size = 10000
valid_ds_size = 200
context_length = 128
tokenizer_ckpt = "huggingface-course/code-search-net-tokenizer"
model_checkpoint = "gpt-2"
push_to_hub=False
train_bs = 32
eval_bs = 32
eval_strategy="no"
save_strategy="no"
gacc_steps = 8
num_train_epochs = 1
lr_scheduler_type="cosine"
lr = 5e-4
task = "clm"
ckpt_name = f"{model_checkpoint.split("/")[-1]}-xxsmall-pretrained-{task}-{dataset_dict['path'].split("/")[-1]}"
print(ckpt_name)

# Gathering the data

- Python code is abundantly available from code repositories such as GitHub, which we can use to create a dataset by scraping for every Python repository.
- This was the approach taken in the [Transformers textbook](https://learning.oreilly.com/library/view/natural-language-processing/9781098136789/) to pretrain a large GPT-2 model. Using a GitHub dump of about 180 GB containing roughly 20 million Python files called `codeparrot`, the authors built a dataset that they then shared on the [Hugging Face Hub](https://huggingface.co/datasets/transformersbook/codeparrot).
- However, training on the full corpus is time- and compute-consuming, and we only need the subset of the dataset concerned with the Python data science stack.
    - So, let’s start by filtering the `codeparrot` dataset for all files that include any of the libraries in this stack.
    - Because of the dataset’s size, we want to avoid downloading it; instead, we’ll use the streaming feature to filter it on the fly.

In [ ]:
# # This cell will take a very long time to execute, so you should skip it and go to
# # the next one!

# def any_keyword_in_string(string, keywords):
#     for keyword in keywords:
#         if keyword in string:
#             return True
#     return False


# def filter_streaming_dataset(dataset, filters):
#     filtered_dict = defaultdict(list)
#     total = 0
#     for sample in tqdm(iter(dataset)):
#         total += 1
#         if any_keyword_in_string(sample["content"], filters):
#             for k, v in sample.items():
#                 filtered_dict[k].append(v)
#     print(f"{len(filtered_dict['content'])/total:.2%} of data after filtering.")
#     return Dataset.from_dict(filtered_dict)

# split = "train"  # "valid"
# filters = ["pandas", "sklearn", "matplotlib", "seaborn"]

# data = load_dataset(f"transformersbook/codeparrot-{split}", split=split, streaming=True)
# filtered_data = filter_streaming_dataset(data, filters)

In [ ]:
ds_train = load_dataset(f"{dataset_dict["path"]}-train", split="train")
ds_valid = load_dataset(f"{dataset_dict["path"]}-valid", split="validation")

# TODO: remove shuffle and use full size if resources allow
raw_datasets = DatasetDict(
    {
        "train": ds_train.shuffle().select(range(train_ds_size)),
        "valid": ds_valid.shuffle().select(range(valid_ds_size))
    }
)

print(raw_datasets)
print("="*100)
for key in raw_datasets["train"][0]:
    print(f"{key.upper()}: {raw_datasets['train'][0][key][:200]}")

# Preparing the dataset

- The first step will be to tokenize the data, so we can use it for training.
- Since our goal is to mainly autocomplete short function calls, we can keep the context size relatively small.
    - This has the benefit that we can train the model much faster and it requires significantly less memory.
- If it is important for your application to have more context, make sure you increase that number, but also keep in mind that this comes with a greater GPU memory footprint.
- For now, let’s fix the context size at 128 tokens, as opposed to the 1,024 or 2,048 used in GPT-2 or GPT-3, respectively.
- Most documents contain many more than 128 tokens, so simply truncating the inputs to the maximum length would eliminate a large fraction of our dataset. Instead,
    - we’ll use the `return_overflowing_tokens` option to tokenize the whole input and split it into several chunks
    - We’ll also use the `return_length` option to return the length of each created chunk automatically.
    - Often the last chunk will be smaller than the context size, and we’ll get rid of these pieces to avoid padding issues
        - we don’t really need them as we have plenty of data anyway.

![](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter7/chunking_texts-dark.svg)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(tokenizer_ckpt)

outputs = tokenizer(
    raw_datasets["train"][:2]["content"],
    truncation=True,
    max_length=context_length,
    return_overflowing_tokens=True,
    return_length=True,
)

print(f"Input IDs length: {len(outputs['input_ids'])}")
print(f"Input chunk lengths: {(outputs['length'])}")
print(f"Chunk mapping: {outputs['overflow_to_sample_mapping']}")

- We can see that we get 34 segments in total from those two examples.
- Looking at the chunk lengths, we can see that the chunks at the ends of both documents have fewer than 128 tokens (69 and 65, respectively).
- These represent just a small fraction of the total chunks that we have, so we can safely throw them away.
- With the `overflow_to_sample_mapping` field, we can also reconstruct which chunks belonged to which input samples.

#### TODO
- Getting rid of all the chunks that are smaller than the context size isn’t a big issue here because we’re using small context windows.
- As you increase the context size (or if you have a corpus of short documents), the fraction of chunks that are thrown away will also grow.
- A more efficient way to prepare the data is to join all the tokenized samples in a batch with an `eos_token_id` token in between, and then perform the chunking on the concatenated sequences.
- As an exercise, modify the `tokenize()` function to make use of that approach.
    - Note that you’ll want to set `truncation=False` and remove the other arguments from the tokenizer to get the full sequence of token IDs.

In [ ]:
def tokenize(batch, tokenizer):
    outputs = tokenizer(
        batch["content"],
        truncation=True,
        max_length=context_length,
        return_overflowing_tokens=True,
        return_length=True,
    )
    return {
        "input_ids": [
            ids for l, ids in zip(outputs["length"], outputs["input_ids"])
            if l == context_length
        ]
    }

tokenized_datasets = raw_datasets.map(
    tokenize,
    batched=True,
    fn_kwargs = {"tokenizer": tokenizer},
    remove_columns=raw_datasets["train"].column_names
)

In [ ]:
print(tokenized_datasets)
print("=======================")
print(f"total number of train tokens: {len(tokenized_datasets["train"]) * context_length / 1e9} B")

- For reference, OpenAI’s GPT-3 and Codex models are trained on 300 and 100 billion tokens, respectively, where the Codex models are initialized from the GPT-3 checkpoints.
- Our goal in this section is not to compete with these models, which can generate long, coherent texts, but to create a scaled-down version providing a quick autocomplete function for data scientists.

# Train

## Initializing a new model

- Our first step is to freshly initialize a GPT-2 model.
- We’ll use the same configuration for our model as for the small GPT-2 model
- So, we load the pretrained configuration, make sure that the tokenizer size matches the model vocabulary size and pass the `bos` and `eos` (beginning and end of sequence) token IDs
- Note that this is the first time we don’t use the `from_pretrained()` function, since we’re actually initializing a model ourself

In [ ]:
config = AutoConfig.from_pretrained(
    "gpt2",
    vocab_size=len(tokenizer),
    n_ctx=context_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

model = GPT2LMHeadModel(config)
model_size = sum(t.numel() for t in model.parameters())
print(f"GPT-2 size: {model_size/1e6:.1f}M parameters")

## Data collation

- We can use the `DataCollatorForLanguageModeling` collator, which is designed specifically for language modeling.
- Besides stacking and padding batches, it also takes care of creating the language model labels
    - in causal language modeling the inputs serve as labels too (just shifted by one element),
    - and this data collator creates them on the fly during training so we don’t need to duplicate the `input_ids`.
        - ⚠️ Shifting the inputs and labels to align them happens inside the model, so the data collator just copies the inputs to create the labels.
- Note that `DataCollatorForLanguageModeling` supports both masked language modeling (MLM) and causal language modeling (CLM).
    - By default it prepares data for MLM, but we can switch to CLM by setting the argument `mlm=False`

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(
    tokenizer,
    mlm=False,
    pad_to_multiple_of=8
)

- Since we only keep examples with `length == context_length`, every example already has the same length.
- In that case, `DataCollatorForLanguageModeling` is doing extra work mainly to copy labels.
- A simpler collator can reduce a bit of CPU overhead.

In [ ]:
# def data_collator(features):
#     input_ids = torch.tensor([f["input_ids"] for f in features], dtype=torch.long)
#     return {
#         "input_ids": input_ids,
#         "labels": input_ids.clone(),
#     }

In [ ]:
out = data_collator([tokenized_datasets["train"][i] for i in range(5)])
for key in out:
    print(f"{key} shape: {out[key].shape}")

## Trainer

- We’ll use a cosine learning rate schedule with some warmup
- and an effective batch size of 256 (`per_device_train_batch_size` \* `gradient_accumulation_steps`).
    - Gradient accumulation is used when a single batch does not fit into memory, and incrementally builds up the gradient through several forward/backward passes.

In [ ]:
eval_steps = round(len(tokenized_datasets["train"]) // (train_bs * gacc_steps * 10), -2)
print(eval_steps)

In [ ]:
args = TrainingArguments(
    output_dir=ckpt_name,
    per_device_train_batch_size=train_bs,
    per_device_eval_batch_size=eval_bs,
    push_to_hub=push_to_hub,
    gradient_accumulation_steps=gacc_steps,
    num_train_epochs=num_train_epochs,
    lr_scheduler_type=lr_scheduler_type,
    learning_rate=lr,
    eval_strategy=eval_strategy,
    save_strategy=save_strategy,
    eval_steps=eval_steps,
    logging_steps=eval_steps,
    save_steps=eval_steps,
    warmup_steps=eval_steps // 5,
    dataloader_num_workers=os.cpu_count(),
    fp16=True,
    bf16=False,
    dataloader_pin_memory=True,
    weight_decay=0.1,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    processing_class=tokenizer,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["valid"],
)

In [ ]:
trainer.train()
# TODO: this is too slow. Find a way to make it run faster

In [ ]:
# trainer.push_to_hub()

# Inference | Code generation with a pipeline

In [ ]:
import torch
from transformers import pipeline

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
pipe = pipeline(
    "text-generation", model="huggingface-course/codeparrot-ds", device=device
)

In [ ]:
txt = """\
# create some data
x = np.random.randn(100)
y = np.random.randn(100)

# create scatter plot with x, y
"""
print(pipe(txt, num_return_sequences=1)[0]["generated_text"])

In [ ]:
txt = """\
# create some data
x = np.random.randn(100)
y = np.random.randn(100)

# create dataframe from x and y
"""
print(pipe(txt, num_return_sequences=1)[0]["generated_text"])

In [ ]:
txt = """\
# dataframe with profession, income and name
df = pd.DataFrame({'profession': x, 'income':y, 'name': z})

# calculate the mean income per profession
"""
print(pipe(txt, num_return_sequences=1)[0]["generated_text"])

In [ ]:
txt = """
# import random forest regressor from scikit-learn
from sklearn.ensemble import RandomForestRegressor

# fit random forest model with 300 estimators on X, y:
"""
print(pipe(txt, num_return_sequences=1)[0]["generated_text"])